# 08. 고급 서브플롯 구성

## 학습 목표
- plt.subplots()로 복합 레이아웃 구성
- GridSpec으로 비대칭 패널 배치
- 공유 축(sharex/sharey) 활용
- 대시보드형 멀티 차트 구성

---

## 1. 왜 서브플롯이 필요한가?

**제조업 실무 사례:**
- 다양한 KPI를 한 화면에 통합 표시
- 여러 제품 라인의 성능 동시 비교
- 원인-결과 관계를 나란히 시각화

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from matplotlib.gridspec import GridSpec

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
# 데이터 로드
mpg = sns.load_dataset('mpg')
tips = sns.load_dataset('tips')
iris = sns.load_dataset('iris')

## 2. 기본 서브플롯: 2x2 그리드

**비즈니스 질문:** 자동차의 4가지 속성을 한눈에 비교하려면?

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 좌상: 연비 분포
axes[0, 0].hist(mpg['mpg'], bins=25, color='#3A86FF', edgecolor='black', alpha=0.7)
axes[0, 0].set_title('연비 분포', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('MPG')
axes[0, 0].set_ylabel('빈도')
axes[0, 0].grid(True, alpha=0.3, axis='y')

# 우상: 무게-연비 관계
axes[0, 1].scatter(mpg['weight'], mpg['mpg'], alpha=0.5, s=50, c='#FF006E', edgecolor='black', linewidth=0.5)
axes[0, 1].set_title('무게 vs 연비', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('무게 (lbs)')
axes[0, 1].set_ylabel('MPG')
axes[0, 1].grid(True, alpha=0.3)

# 좌하: 제조국별 평균 연비
origin_mpg = mpg.groupby('origin')['mpg'].mean().sort_values(ascending=False)
axes[1, 0].barh(origin_mpg.index, origin_mpg.values, color='#06FFA5', edgecolor='black', linewidth=1.5)
axes[1, 0].set_title('제조국별 평균 연비', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('평균 MPG')
axes[1, 0].grid(True, alpha=0.3, axis='x')

# 우하: 연식별 추이
yearly_mpg = mpg.groupby('model_year')['mpg'].mean()
axes[1, 1].plot(yearly_mpg.index, yearly_mpg.values, marker='o', linewidth=2.5, 
                markersize=8, color='#FFBE0B', markeredgecolor='black', markeredgewidth=1)
axes[1, 1].set_title('연식별 평균 연비', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('연식')
axes[1, 1].set_ylabel('평균 MPG')
axes[1, 1].grid(True, alpha=0.3)

fig.suptitle('자동차 연비 종합 분석 대시보드', fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

# 💡 인사이트: 4가지 관점(분포, 관계, 비교, 추세)을 한 화면에 통합 → 의사결정 효율 향상

## 3. 공유 축: sharex와 sharey

**비즈니스 질문:** 실린더별 연비 분포를 정렬된 축으로 비교하려면?

In [ ]:
cylinders = [4, 6, 8]
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)  # Y축 공유

for i, cyl in enumerate(cylinders):
    data = mpg[mpg['cylinders'] == cyl]['mpg']
    axes[i].hist(data, bins=15, color=f'C{i}', edgecolor='black', alpha=0.7)
    axes[i].set_title(f'{cyl}기통', fontsize=13, fontweight='bold')
    axes[i].set_xlabel('MPG', fontsize=11)
    axes[i].grid(True, alpha=0.3, axis='y')
    
    # 평균선 추가
    mean_val = data.mean()
    axes[i].axvline(mean_val, color='red', linestyle='--', linewidth=2.5,
                   label=f'평균: {mean_val:.1f}')
    axes[i].legend(fontsize=10)

axes[0].set_ylabel('빈도', fontsize=11)
fig.suptitle('실린더별 연비 분포 비교 (Y축 공유)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

# 💡 인사이트: Y축 공유로 빈도 차이 명확 → 4기통이 가장 많은 샘플
# 💡 X축 범위가 다름 → 8기통은 고연비 영역 없음

## 4. GridSpec: 비대칭 레이아웃

**고급 기법:** 메인 차트와 보조 차트를 다른 크기로 배치

In [ ]:
fig = plt.figure(figsize=(14, 10))
gs = GridSpec(3, 3, figure=fig)

# 메인 차트 (좌측 2x2)
ax_main = fig.add_subplot(gs[:2, :2])
ax_main.scatter(mpg['weight'], mpg['mpg'], s=mpg['horsepower'], 
                c=mpg['cylinders'], cmap='viridis', alpha=0.6,
                edgecolors='black', linewidth=0.5)
ax_main.set_xlabel('무게 (lbs)', fontsize=12)
ax_main.set_ylabel('연비 (mpg)', fontsize=12)
ax_main.set_title('무게-연비 관계 (크기=마력, 색상=실린더)', fontsize=13, fontweight='bold')
ax_main.grid(True, alpha=0.3)

# 상단 보조 차트 (무게 분포)
ax_top = fig.add_subplot(gs[:2, 2])
ax_top.hist(mpg['weight'], bins=30, orientation='horizontal', 
            color='#FF006E', edgecolor='black', alpha=0.7)
ax_top.set_ylabel('무게 (lbs)', fontsize=11)
ax_top.set_xlabel('빈도', fontsize=11)
ax_top.set_title('무게 분포', fontsize=11, fontweight='bold')
ax_top.grid(True, alpha=0.3, axis='x')

# 하단 보조 차트 (연비 분포)
ax_bottom = fig.add_subplot(gs[2, :2])
ax_bottom.hist(mpg['mpg'], bins=30, color='#3A86FF', edgecolor='black', alpha=0.7)
ax_bottom.set_xlabel('연비 (mpg)', fontsize=11)
ax_bottom.set_ylabel('빈도', fontsize=11)
ax_bottom.set_title('연비 분포', fontsize=11, fontweight='bold')
ax_bottom.grid(True, alpha=0.3, axis='y')

# 우하단 (통계 텍스트)
ax_stats = fig.add_subplot(gs[2, 2])
ax_stats.axis('off')
stats_text = f"""
📊 통계 요약

전체: {len(mpg)}대

평균 연비: {mpg['mpg'].mean():.1f} mpg
평균 무게: {mpg['weight'].mean():.0f} lbs

상관계수:
무게-연비: {mpg['weight'].corr(mpg['mpg']):.3f}
마력-연비: {mpg['horsepower'].corr(mpg['mpg']):.3f}
"""
ax_stats.text(0.1, 0.5, stats_text, fontsize=11, verticalalignment='center',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

fig.suptitle('GridSpec 고급 레이아웃', fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

# 💡 인사이트: 메인 차트와 주변 분포(marginal distribution)를 결합 → 다차원 이해 향상
# 💡 통계 패널로 정량적 정보 보강

## 5. 복합 차트 유형

**비즈니스 질문:** 팁 데이터의 다양한 측면을 통합 분석하려면?

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# 1. 계산서-팁 산점도
axes[0, 0].scatter(tips['total_bill'], tips['tip'], alpha=0.5, s=50, c='#3A86FF')
axes[0, 0].set_title('계산서 vs 팁', fontweight='bold')
axes[0, 0].set_xlabel('총 계산서 ($)')
axes[0, 0].set_ylabel('팁 ($)')
axes[0, 0].grid(True, alpha=0.3)

# 2. 요일별 거래 건수
day_counts = tips['day'].value_counts().reindex(['Thur', 'Fri', 'Sat', 'Sun'])
axes[0, 1].bar(day_counts.index, day_counts.values, color='#FF006E', edgecolor='black')
axes[0, 1].set_title('요일별 거래 건수', fontweight='bold')
axes[0, 1].set_ylabel('건수')
axes[0, 1].grid(True, alpha=0.3, axis='y')

# 3. 팁 비율 분포
tips['tip_pct'] = tips['tip'] / tips['total_bill'] * 100
axes[0, 2].hist(tips['tip_pct'], bins=20, color='#06FFA5', edgecolor='black', alpha=0.7)
axes[0, 2].axvline(tips['tip_pct'].mean(), color='red', linestyle='--', linewidth=2.5)
axes[0, 2].set_title('팁 비율 분포', fontweight='bold')
axes[0, 2].set_xlabel('팁 비율 (%)')
axes[0, 2].grid(True, alpha=0.3, axis='y')

# 4. 시간대별 평균 팁
time_tip = tips.groupby('time')['tip'].mean()
axes[1, 0].barh(time_tip.index, time_tip.values, color='#FFBE0B', edgecolor='black')
axes[1, 0].set_title('시간대별 평균 팁', fontweight='bold')
axes[1, 0].set_xlabel('평균 팁 ($)')
axes[1, 0].grid(True, alpha=0.3, axis='x')

# 5. 인원수별 박스플롯
size_order = sorted(tips['size'].unique())
box_data = [tips[tips['size']==s]['tip'] for s in size_order]
bp = axes[1, 1].boxplot(box_data, labels=size_order, patch_artist=True)
for patch in bp['boxes']:
    patch.set_facecolor('#E63946')
    patch.set_alpha(0.7)
axes[1, 1].set_title('인원수별 팁 분포', fontweight='bold')
axes[1, 1].set_xlabel('인원수')
axes[1, 1].set_ylabel('팁 ($)')
axes[1, 1].grid(True, alpha=0.3, axis='y')

# 6. 성별-흡연 조합 평균 팁
combo_tip = tips.groupby(['sex', 'smoker'])['tip'].mean().unstack()
x = np.arange(len(combo_tip.index))
width = 0.35
axes[1, 2].bar(x - width/2, combo_tip['Yes'], width, label='흡연', color='#FF6B6B')
axes[1, 2].bar(x + width/2, combo_tip['No'], width, label='비흡연', color='#4ECDC4')
axes[1, 2].set_title('성별-흡연 조합별 평균 팁', fontweight='bold')
axes[1, 2].set_xlabel('성별')
axes[1, 2].set_ylabel('평균 팁 ($)')
axes[1, 2].set_xticks(x)
axes[1, 2].set_xticklabels(combo_tip.index)
axes[1, 2].legend()
axes[1, 2].grid(True, alpha=0.3, axis='y')

fig.suptitle('팁 데이터 종합 대시보드', fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

# 💡 인사이트: 6가지 다른 차트 유형으로 다각도 분석 → 포괄적 이해
# 💡 한 화면에서 패턴 발견 및 가설 검증 가능

## 6. 중첩 서브플롯: inset_axes

**고급 기법:** 메인 차트 내부에 확대도 삽입

In [ ]:
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

fig, ax = plt.subplots(figsize=(12, 8))

# 메인 차트: 전체 데이터
scatter = ax.scatter(mpg['weight'], mpg['mpg'], 
                    s=mpg['horsepower'], c=mpg['cylinders'], 
                    cmap='viridis', alpha=0.6, edgecolors='black', linewidth=0.5)

ax.set_xlabel('무게 (lbs)', fontsize=12)
ax.set_ylabel('연비 (mpg)', fontsize=12)
ax.set_title('무게-연비 관계 (메인 + 확대도)', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)

# 컬러바
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('실린더 수', fontsize=11)

# Inset 차트: 경량 고연비 영역 확대
ax_inset = inset_axes(ax, width="35%", height="35%", loc='upper right',
                     bbox_to_anchor=(0, 0.05, 1, 1), bbox_transform=ax.transAxes)

# 경량 고연비 차량만 필터링
light_efficient = mpg[(mpg['weight'] < 2500) & (mpg['mpg'] > 25)]
ax_inset.scatter(light_efficient['weight'], light_efficient['mpg'],
                s=light_efficient['horsepower']*2, c=light_efficient['cylinders'],
                cmap='viridis', alpha=0.8, edgecolors='red', linewidth=1.5)

ax_inset.set_xlabel('무게', fontsize=9)
ax_inset.set_ylabel('연비', fontsize=9)
ax_inset.set_title('경량 고연비 구간 확대', fontsize=10, fontweight='bold', color='red')
ax_inset.grid(True, alpha=0.3)

# 확대 영역 표시
from matplotlib.patches import Rectangle
rect = Rectangle((1500, 25), 1000, 25, linewidth=2, edgecolor='red', 
                facecolor='none', linestyle='--')
ax.add_patch(rect)

plt.tight_layout()
plt.show()

# 💡 인사이트: Inset으로 특정 영역 강조 → 이상치나 관심 구간 집중 분석
# 💡 경량 고연비 차량은 대부분 4기통

---

## 📊 핵심 요약

### 코드 패턴 (30%)
```python
# 기본 서브플롯
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(12, 10))
axes[0, 0].plot(x, y)  # 인덱싱으로 접근

# 공유 축
fig, axes = plt.subplots(1, 3, sharey=True)

# GridSpec
from matplotlib.gridspec import GridSpec
fig = plt.figure(figsize=(12, 8))
gs = GridSpec(3, 3)
ax1 = fig.add_subplot(gs[:2, :2])  # 2x2 영역

# Inset
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
ax_inset = inset_axes(ax, width="30%", height="30%", loc='upper right')
```

### 레이아웃 선택 가이드 (70%)
| 목적 | 도구 | 적용 사례 |
|------|------|----------|
| 균등 비교 | subplots | 제품 라인별 성능 비교 |
| 축 정렬 | sharex/sharey | 시계열 여러 지표 동시 표시 |
| 비대칭 레이아웃 | GridSpec | 메인 차트 + 보조 차트 |
| 상세 강조 | inset_axes | 특정 구간 확대 |

### 도출된 인사이트
1. **대시보드 효과**: 6개 차트 조합으로 팁 패턴 전방위 분석
2. **공유 축 장점**: Y축 통일로 실린더별 빈도 차이 명확 비교
3. **GridSpec 활용**: 메인 scatter + 주변 hist로 2D 분포 완전 이해
4. **Inset 효과**: 경량 고연비 구간 확대 → 4기통 집중 발견

### 실무 적용
- **경영진 보고**: GridSpec으로 핵심 지표 대시보드
- **품질 모니터링**: 공유 축으로 여러 라인 동시 추적
- **이상 분석**: Inset으로 특이 구간 강조

### 주의사항
- 서브플롯 너무 많으면(>9개) 가독성 저하 → 핵심만 선택
- 공유 축 사용 시 척도 차이 큰 변수는 부적합
- GridSpec 복잡도 증가 → 코드 주석 필수
- tight_layout() 필수 → 레이블 겹침 방지